# 02 Data Preprocessing

This notebook is part of a reproducible machine learning project using the BRFSS 2015 diabetes health indicators dataset. All outputs are saved automatically to the `results` directory.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from project_utils import *
set_reproducibility()

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

Project root: /Users/guuuuu/Desktop/final0519
Dataset path: /Users/guuuuu/Desktop/final0519/diabetes_012_health_indicators_BRFSS2015.csv


## Purpose

This notebook prepares the data for machine learning. It applies duplicate removal, stratified train/test splitting, feature scaling with `StandardScaler`, and class balancing with SMOTE. Stratification preserves the original target proportions in both training and test sets. SMOTE is applied only to the training data to avoid test-set leakage.

In [2]:
raw_df = load_raw_data()
clean_df = clean_data(raw_df, remove_duplicates=True)
X_train, X_test, y_train, y_test, features = split_data(clean_df)

split_summary = pd.DataFrame({
    "Split": ["Training", "Test"],
    "Rows": [len(X_train), len(X_test)],
    "Feature columns": [len(features), len(features)]
})
save_table(split_summary, "train_test_split_summary")
display(split_summary)

,Split,Rows,Feature columns
0,Training,183824,21
1,Test,45957,21


## Standardization and SMOTE

`StandardScaler` gives continuous and ordinal variables comparable numeric scale. SMOTE synthesizes minority-class training examples, improving the model's opportunity to learn prediabetes and diabetes patterns instead of mainly optimizing for the majority class.

In [3]:
X_train_scaled, X_test_scaled = create_preprocessed_artifacts(clean_df)
display(pd.read_csv(TABLES_DIR / "smote_class_distribution.csv"))
print("Preprocessing artifacts saved.")

,Class,Count_Before_SMOTE,Count_After_SMOTE
0,No diabetes,152043,152043
1,Prediabetes,3703,152043
2,Diabetes,28078,152043


Preprocessing artifacts saved.


## Feature Engineering Notes

The BRFSS dataset already provides clinically meaningful engineered indicators, including blood pressure, cholesterol, BMI, physical activity, general health, age group, education, and income. We preserve the original interpretable feature names so that SHAP and feature-importance results remain useful for public health interpretation.

In [4]:
feature_info = pd.DataFrame({
    "Feature": features,
    "Role": ["Predictor"] * len(features),
    "Preprocessing": ["StandardScaler"] * len(features)
})
save_table(feature_info, "feature_preprocessing_plan")
display(feature_info)

,Feature,Role,Preprocessing
0,HighBP,Predictor,StandardScaler
1,HighChol,Predictor,StandardScaler
2,CholCheck,Predictor,StandardScaler
3,BMI,Predictor,StandardScaler
4,Smoker,Predictor,StandardScaler
5,Stroke,Predictor,StandardScaler
6,HeartDiseaseorAttack,Predictor,StandardScaler
7,PhysActivity,Predictor,StandardScaler
8,Fruits,Predictor,StandardScaler
9,Veggies,Predictor,StandardScaler
